1. Librerías

In [74]:
%%capture
!pip install langchain_community
!pip install langchain_unstructured
!pip install unstructured
!pip install boto3
!pip install langchain_google_genai
!pip install faiss-cpu
!pip install langchain_openai

In [35]:
from google.colab import userdata

In [3]:

aws_access_key = userdata.get('AWS_ACCESS_KEY')
aws_secret_key = userdata.get('AWS_SECRET_KEY')

In [4]:
BUCKET = "langchain-testing-660463065978-us-east-1-an"

In [5]:
import langchain_community

In [6]:
from langchain_community.document_loaders import (
    TextLoader,
    UnstructuredMarkdownLoader,
    CSVLoader,
    BSHTMLLoader,
    PyPDFLoader
)

2. Loader

In [7]:
from pathlib import Path

In [8]:
import langchain_unstructured

In [9]:
from langchain_unstructured import UnstructuredLoader

In [10]:
import boto3
import tempfile
import os

In [11]:
def pick_loader(path):
    ext = Path(path).suffix.lower()
    if ext == ".txt":
        return TextLoader(path, encoding="utf-8")
    if ext == ".md":
        return UnstructuredMarkdownLoader(path)
    if ext == ".pdf":
        return PyPDFLoader(path)
    if ext in {".html", ".htm"}:
        return BSHTMLLoader(path)
    if ext == ".csv":
        return CSVLoader(path)
    return UnstructuredLoader([path])

In [12]:
s3 = boto3.client(
    "s3",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
)

In [13]:
data = []
with tempfile.TemporaryDirectory() as tmpdir:
    for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents"):
        key = obj["Key"]
        path = os.path.join(tmpdir, os.path.basename(key))
        s3.download_file(BUCKET, key, path)
        data.extend(pick_loader(path).load())

In [14]:
print(data[4].page_content)

ElectroRuta - FAQ de soporte al cliente
Versión 2026-04

1. ¿Por qué mi tarjeta RFID fue rechazada?
Las causas más comunes son token vencido, tarjeta no asociada a una cuenta activa o falla temporal del lector. Si el error ocurrió en MTY-042 o GDL-018 y el equipo era un ER-DC180, verificar si hubo incidencia reciente de firmware antes de pedir reemplazo de tarjeta.

2. ¿Puedo iniciar la carga desde la app aunque mi tarjeta falle?
Sí. En la mayoría de los casos la app usa autorización en línea y puede funcionar aun cuando el lector RFID esté degradado.

3. ¿Cómo se calculan las tarifas?
El cobro combina tarifa base por kWh y, según el sitio, fee de inactividad e incremento nocturno. Los sitios para flotillas pueden tener tarifa distinta a la de usuarios residenciales.

4. ¿Qué significa fee de inactividad?
Es un cargo por ocupar el conector después de haber terminado la carga. En DC rápido se empieza a contar 10 minutos después del fin de sesión; en AC normalmente son 20 minutos.

5. ¿S

3. Chunks

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
text_chunks = text_splitter.split_documents(data)

In [17]:
len(text_chunks)

155

In [19]:
text_contents = [chunk.page_content for chunk in text_chunks]

In [20]:
text_contents[0] == text_chunks[0].page_content

True

In [37]:
len(text_contents)

155

4. Embeddings

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [22]:
import time
from time import sleep

In [23]:
from langchain_community.vectorstores import FAISS

In [24]:
GEMINI_KEY = userdata.get('GEMINI_KEY')

In [25]:
client = genai.Client(api_key=GEMINI_KEY)

In [26]:
batch_size = 100
batches = [text_contents[i:i + batch_size] for i in range(0, len(text_contents), batch_size)]

In [60]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_KEY)

In [63]:
embeddings = []

for i in range(0, len(batches)):
    result = embedding_model.embed_documents(batches[i])
    embeddings.extend(result)
    # para el ultimo indice no necesita esperar 60 segundos
    if i == len(batches) - 1:
        break
    time.sleep(60)

In [64]:
text_embeddings = list(zip(text_contents, embeddings))

In [65]:
documentSearch = FAISS.from_embeddings(
    text_embeddings=text_embeddings,
    embedding=embedding_model
)

In [70]:
len(embeddings[0])

3072

In [83]:
documentSearch.save_local("/faiss_index")

5. Retrieval

Pendiente de implementar correctamente las chains

In [75]:
from langchain_openai import ChatOpenAI

In [90]:
KEY = userdata.get('GROQ_API_KEY')

In [84]:
query = "Hi, who are you? I am speaking to a machine in the middle of my hometown, that's weird, why is that possible? I think you are a local model and you are processing my voice to text"


In [82]:
documentSearch.similarity_search_with_score("Hi, who are you? I am speaking to a machine in the middle of my hometown, that's weird, why is that possible? I think you are a local model and you are processing my voice to text")[0]

(Document(id='30db8b47-721a-41f7-9a22-e3642ca41a9b', metadata={}, page_content='status: available\nnotes: Sin anomalías relevantes'),
 np.float32(0.74992454))

In [89]:
documentSearch.similarity_search_with_score("De qué trata el documento?")

[(Document(id='021650ac-69ee-405d-9bfe-156b3386c1c1', metadata={}, page_content='ElectroRuta - Manual operativo de estaciones\nDocumento interno para operación N1 y mantenimiento ligero | Vigente desde 2026-04-01\n1. Alcance del documento'),
  np.float32(0.50923884)),
 (Document(id='ba5703c8-ef3a-437a-9d4a-eeecb9b444d5', metadata={}, page_content='"pantalla rara".\nVersión del documento: 2026.04. Operación revisada por Field Reliability y FleetOps LATAM.'),
  np.float32(0.55678356)),
 (Document(id='1645f843-33ac-4906-86c9-523032fb0ada', metadata={}, page_content='Bienvenido al programa ElectroRuta Fleet. Esta guía resume el alta inicial, los datos que debes preparar y las decisiones que conviene tomar antes de conectar vehículos, tarjetas RFID o integraciones'),
  np.float32(0.5603532)),
 (Document(id='71e12125-f020-49a5-a661-f6297d7dc8aa', metadata={}, page_content='ElectroRuta API para socios de flotillas\n\nVersión: v1.8 Última actualización: 2026-04-08 Ámbito: Integración con porta

In [85]:

llm = ChatOpenAI(
    model_name="openai/gpt-oss-20b",  # Changed to a Groq-supported model
    temperature=0.5,
    openai_api_key=KEY,
    base_url="https://api.groq.com/openai/v1" # Explicitly setting Groq API base_url
)

In [86]:
print(llm.invoke(query).content)

Hi there! I’m ChatGPT, an AI language model created by OpenAI. I don’t run on your local machine; instead, I live in the cloud. When you speak, the voice‑to‑text part of the app or device you’re using turns your words into text, sends that text to the server where I am, and then I generate a written reply. That reply comes back to you as text (or spoken, if the app reads it aloud).

So the “weird” part is just the combination of a local voice‑recognition engine and a remote language model. If you have any more questions about how it works—or anything else—just let me know!


In [95]:
!pip install langchain

In [97]:
from langchain.chains.question_answering import load_qa_chain

ModuleNotFoundError: No module named 'langchain.chains'

In [91]:

# Ahora la funcion ya esta disponible en el entorno
chain = load_qa_chain(llm, chain_type="stuff")

NameError: name 'load_qa_chain' is not defined

In [98]:
docs = documentSearch.similarity_search(query)

In [ ]:
response = chain.invoke({"input_documents": docs, "question": query})